<a href="https://colab.research.google.com/github/deepak8186863620/Deep_Research_Analysis/blob/main/WorkingOfRankingAlgorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q sentence-transformers numpy pandas scikit-learn rank-bm25

In [4]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
print(" libraries download successfull")

 libraries download successfull


In [6]:
query="how does the Retrievel Augmented generation helps the Large langauage models?"
print(" The query:")
print(query)



 The query:
how does the Retrievel Augmented generation helps the Large langauage models?


In [7]:
import requests
import xml.etree.ElementTree as ET
url = "https://export.arxiv.org/api/query"
params = {
    "search_query": "all:retrieval augmented generation",
    "start": 0,
    "max_results": 20
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

Status code: 200


In [9]:
root = ET.fromstring(response.text)

entries = root.findall("{http://www.w3.org/2005/Atom}entry")

print("Number of papers retrieved:", len(entries))

Number of papers retrieved: 20


In [10]:
papers = []

for entry in entries:
    title = entry.find("{http://www.w3.org/2005/Atom}title").text
    abstract = entry.find("{http://www.w3.org/2005/Atom}summary").text
    published = entry.find("{http://www.w3.org/2005/Atom}published").text
    paper_id = entry.find("{http://www.w3.org/2005/Atom}id").text

    papers.append({
        "title": title.strip(),
        "abstract": abstract.strip(),
        "published": published,
        "paper_id": paper_id
    })

df = pd.DataFrame(papers)

print("Total papers:", len(df))
df.head()

Total papers: 20


,title,abstract,published,paper_id
0,AR-RAG: Autoregressive Retrieval Augmentation ...,We introduce Autoregressive Retrieval Augmenta...,2025-06-08T01:33:05Z,http://arxiv.org/abs/2506.06962v3
1,Intelligent Interaction Strategies for Context...,Human cognition is constrained by processing l...,2025-04-18T13:35:21Z,http://arxiv.org/abs/2504.13684v1
2,Factually: Exploring Wearable Fact-Checking fo...,Wearable devices are transforming human capabi...,2025-04-24T02:29:50Z,http://arxiv.org/abs/2504.17204v1
3,Designing AI Systems that Augment Human Perfor...,The recent rapid advancement of LLM-based AI s...,2025-04-20T17:40:28Z,http://arxiv.org/abs/2504.14689v1
4,Automated Literature Review Using NLP Techniqu...,This research presents and compares multiple a...,2024-11-27T18:27:07Z,http://arxiv.org/abs/2411.18583v1


In [11]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [13]:
query_embedding = model.encode([query])

paper_embeddings = model.encode(
    df["abstract"].tolist()
)

print("Query embedding shape:", query_embedding.shape)
print("Paper embeddings shape:", paper_embeddings.shape)

Query embedding shape: (1, 384)
Paper embeddings shape: (20, 384)


In [14]:
similarities = cosine_similarity(
    query_embedding,
    paper_embeddings
)[0]

df["relevance_score"] = similarities

df[["title", "relevance_score"]].sort_values(
    by="relevance_score",
    ascending=False
).head(10)

,title,relevance_score
18,Reconstructing Context: Evaluating Advanced Ch...,0.561344
6,EVOR: Evolving Retrieval for Code Generation,0.521295
8,Ragas: Automated Evaluation of Retrieval Augme...,0.493397
14,RAGPart & RAGMask: Retrieval-Stage Defenses Ag...,0.456650
16,IGMiRAG: Intuition-Guided Retrieval-Augmented ...,0.437372
7,Riddle Me This! Stealthy Membership Inference ...,0.435141
11,FAIR-RAG: Faithful Adaptive Iterative Refineme...,0.434007
4,Automated Literature Review Using NLP Techniqu...,0.428570
17,CARROT: A Learned Cost-Constrained Retrieval O...,0.414580
0,AR-RAG: Autoregressive Retrieval Augmentation ...,0.412914


#**the below is the changed index and the above code give us the correct one but the indexing  in the above was based on the first 20 per and their score**

In [15]:
ranked_df = df[["title", "relevance_score"]].sort_values(
    by="relevance_score",
    ascending=False
).reset_index(drop=True)

ranked_df.index = ranked_df.index + 1

ranked_df.head(10)

,title,relevance_score
1,Reconstructing Context: Evaluating Advanced Ch...,0.561344
2,EVOR: Evolving Retrieval for Code Generation,0.521295
3,Ragas: Automated Evaluation of Retrieval Augme...,0.493397
4,RAGPart & RAGMask: Retrieval-Stage Defenses Ag...,0.456650
5,IGMiRAG: Intuition-Guided Retrieval-Augmented ...,0.437372
6,Riddle Me This! Stealthy Membership Inference ...,0.435141
7,FAIR-RAG: Faithful Adaptive Iterative Refineme...,0.434007
8,Automated Literature Review Using NLP Techniqu...,0.428570
9,CARROT: A Learned Cost-Constrained Retrieval O...,0.414580
10,AR-RAG: Autoregressive Retrieval Augmentation ...,0.412914


In [16]:
import requests

title = df.loc[0, "title"]

url = "https://api.openalex.org/works"

params = {
    "search": title,
    "per-page": 1
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

Status code: 200
